***Character-Level Tokenizer (stoi/itos/BOS)***

In [ ]:
class CharTokenizer:
	def __init__(self, text: str):
		
		special = ['<BOS>', '<EOS>']
		chars = sorted(set(text))
		vocab = special + chars  # ['<BOS>', '<EOS>','e','h','l','o']
		self.stoi = {ch: i for i, ch in enumerate(vocab)}   # char → index || {'<BOS>':0, '<EOS>':1,'e':2,'h':3,'l':4,'o':5}
		self.itos = {i: ch for i, ch in enumerate(vocab)}   # index → char || {0:'<BOS>', 1:'<EOS>',2:'e',3:'h',4:'l',5:'o'}
		self.vocab_size = len(vocab)

	def encode(self, text: str) -> list:
	
		return [self.stoi['<BOS>']] + [self.stoi[ch] for ch in text] + [self.stoi['<EOS>']]
		
		
	def decode(self, indices: list) -> str:
		
		return "".join([self.itos[i] for i in indices])

tok = CharTokenizer('hello')
print(tok.vocab_size)
print("-----------------------")
tok = CharTokenizer('hello')
print(tok.encode('hello'))
print("-----------------------")
tok = CharTokenizer('hello')
print(tok.decode([0, 3, 2, 4, 4, 5, 1]))

6
-----------------------
[0, 3, 2, 4, 4, 5, 1]
-----------------------
<BOS>hello<EOS>


***Build Bigram Count Matrix from Words***

In [ ]:
import numpy as np 

def bigram_counts(words):
	
	all_chars = sorted(set(''.join(words)))
	vocab = ['.'] + all_chars
	stoi = {ch: i for i, ch in enumerate(vocab)}
	N = np.zeros((len(vocab), len(vocab)), dtype=np.int32)
	
	for w in words:
		sequence = ['.'] + list(w) + ['.']
		# print(sequence) 
		for t in range(len(sequence) - 1):
			N[stoi[sequence[t]], stoi[sequence[t+1]]] += 1
	
	return N.tolist()

print(bigram_counts(['ab']))
print("--------------")
print(bigram_counts(['emma']))

['.', 'a', 'b', '.']
[[0, 1, 0], [0, 0, 1], [1, 0, 0]]
--------------
['.', 'e', 'm', 'm', 'a', '.']
[[0, 0, 1, 0], [1, 0, 0, 0], [0, 0, 0, 1], [0, 1, 0, 1]]


***Min-Max Scaling of Feature Values***

In [ ]:
def min_max(x: list[float]) -> list[float]:
	
	_min = min(x)
	_max = max(x)
	result = []
	for num in x :
		norml = (num - _min) / (_max - _min)
		result.append(norml)
	return result 

print(min_max([1, 2, 3, 4, 5]))
print("-----------------------")
print([round(x, 4) for x in min_max([30, 45, 56, 70, 88])])


[0.0, 0.25, 0.5, 0.75, 1.0]
-----------------------
[0.0, 0.2586, 0.4483, 0.6897, 1.0]


***Simulate Markov Chain Transitions***

In [ ]:
import numpy as np
def simulate_markov_chain(transition_matrix, initial_state, num_steps):

	current_state = initial_state  # current_state = 0 || current_state = 1 ||  

	states = [current_state]  # [0,0,1]

	for _ in range(num_steps): # num_steps = 3 || step 0 || step 1 || step 2

		probabilities = transition_matrix[current_state] # transition_matrix[0] = [0.8, 0.2] || transition_matrix[1] = [0.3, 0.7]

		next_state = np.random.choice(transition_matrix.shape[0],p=probabilities) # next_state = ([0,1] , [0.8, 0.2] ) 
																				  # next_state = ([0,1] , [0.3, 0.7] ) 
		states.append(next_state) # [0, 1,  ]

		current_state = next_state # current_state = 0 || current_state = 1 || 

	return states

np.random.seed(42)
transition_matrix = np.array([[0.8, 0.2], [0.3, 0.7]])
print(simulate_markov_chain(transition_matrix, 0, 3))

[0, 0, 1, 1]


***Sample Names from a Bigram Language Model***

In [ ]:
import numpy as np

def sample_name(P, itos, seed, max_len=100):
	# Your code here
	rng = np.random.default_rng(seed) 
	ix = 0
	out = []
	probs = []

	for row in P:
		sum_ = np.sum(row)

		if sum_ == 0:
			zero_row = np.zeros(P.shape[0])
			zero_row[0] = 1
			probs.append(zero_row)
		else:
			probs.append(row / sum_)

	probs = np.array(probs)

	for _ in range(max_len):
		row = probs[ix] 
		r = rng.random()
		cdf = np.cumsum(row)
		j = np.searchsorted(cdf, r)

		if j == 0:
			break 
		else :
			out.append(itos[j])
			ix = j
	return ''.join(out)

P = [[0,1,0,0],[0,0,1,0],[0,0,0,1],[1,0,0,0]]
itos = ['.', 'a', 'b', 'c']
print(sample_name(P, itos, seed=7))
print("------------")
P = [[0,0,1],[1,0,0],[0,1,0]]
itos = ['.', 'a', 'b','d','a','l','l','a','h']
print(sample_name(P, itos, seed=1))

abc
------------
ba


***Row-Normalize a Count Matrix to Probabilities***

In [19]:
import numpy as np

def row_normalize(counts: list[list[float]]) -> list[list[float]]:
	"""Convert a count matrix into a row-stochastic probability matrix."""
	counts = np.array(counts)
	row_sums = counts.sum(axis=1, keepdims=True)  # sum of each row 
	safe_row_sums = row_sums.copy()
	safe_row_sums[row_sums == 0] = 1
	result = counts / safe_row_sums 
	return result.tolist()

print(row_normalize([[1, 1], [2, 2]]))

[[0.5, 0.5], [0.5, 0.5]]


***Vector Element-wise Sum***

In [2]:
def vector_sum(a: list[int|float], b: list[int|float]) -> list[int|float]:
	# Return the element-wise sum of vectors 'a' and 'b'.
	# If vectors have different lengths, return -1.
	q , p = len(a) , len(b)
	if q != p :
		return -1
	else :
		return [a + b for a , b in zip(a,b)] 
	
print(vector_sum([1, 2], [1, 2, 3]))
print("-----------------")
print(vector_sum([1.5, 2.5, 3.0], [2, 1, 4]))

-1
-----------------
[3.5, 3.5, 7.0]


***Average Negative Log-Likelihood for Bigram Model***

In [9]:
import numpy as np

def bigram_avg_nll(P, words, stoi):
    """Average negative log-likelihood of words under a bigram model."""
    N = 0
    log_likelihood  = 0.0
    for word in words :  # word = ['ab']
        
        chs = ["."] + list((word)) + ['.']  # chs = ['.ab.']
        for ch1, ch2 in zip(chs, chs[1:]):   # zip(chs, chs[1:]) => ((.,a),(a,b),(b,.)) => ch1=.,ch2=a | ch1=a,ch2=b | ch1=b,ch2=.
           #print("ch1:",ch1 ,"ch2:",ch2)
           p = P[stoi[ch1], stoi[ch2]] 
           #print(p)   # p = P[0,1] = 0.6 | p = P[1,2] = 0.4 | p = P[2,0] = 0.5
           log_likelihood += np.sum(np.log(p))
           N += 1
    if N == 0:
        return 0.0
    return round(-log_likelihood / N, 4)


stoi = {'.':0,'a':1,'b':2}
P = np.array([[0.1,0.6,0.3],[0.4,0.2,0.4],[0.5,0.3,0.2]])
print(bigram_avg_nll(P, ['ab'], stoi))
print("------------")
stoi = {'.':0,'a':1}
P = np.array([[0.5,0.5],[0.5,0.5]])
print(bigram_avg_nll(P, ['a','aa'], stoi))

0.7068
------------
0.6931


***Laplace Smoothing for Bigram Probabilities***

In [ ]:
import numpy as np

def smooth_bigram_probs(N, k):

    N = np.array(N)
    smoothed = N + k
    row_sums = np.sum(smoothed, axis=1, keepdims=True)
    P = smoothed / row_sums
    return  P.tolist()

N = [[0, 0, 0], [5, 0, 5], [1, 2, 3]]
P = smooth_bigram_probs(N, 1)
print([[round(x, 5) for x in row] for row in P])
print("------------------")
N = [[10, 0], [0, 10]]
P = smooth_bigram_probs(N, 5)
print([[round(x, 5) for x in row] for row in P])

[[0.33333, 0.33333, 0.33333], [0.46154, 0.07692, 0.46154], [0.22222, 0.33333, 0.44444]]
------------------
[[0.75, 0.25], [0.25, 0.75]]


***Softmax Activation Function Implementation***

In [13]:
import math
import numpy as np 

def softmax(scores):

	maxx = np.max(scores)     
	denom = np.sum([math.exp(z - maxx) for z in scores])       # compute once, using ALL scores
	
	results = []
	for score in scores:
		result = float(math.exp(score - maxx) / denom )   
		results.append(result)
	return results

print([round(x, 4) for x in softmax([1, 2, 3])])
print("------------------")
print([round(x, 4) for x in softmax([-1, 0, 5])])

[0.09, 0.2447, 0.6652]
------------------
[0.0025, 0.0067, 0.9909]


***One-Hot Encoding of Nominal Values***


In [15]:
import numpy as np

def to_categorical(x, n_col=None):
	# Your code here
	if not n_col :
		n_col = len(np.unique(x))
		
	output = np.zeros((len(x), n_col))
	for row , column  in enumerate(x):
		output[row][column] = 1
		
	return output 

print(to_categorical(np.array([0, 1, 2, 1, 0])))
print("-------------------")
print(to_categorical(np.array([3, 1, 2, 1, 3]), 4))


[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
-------------------
[[0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]


***Implementation of Log Softmax Function***

In [1]:
import numpy as np

def log_softmax(scores: list) -> np.ndarray:
	# Your code here
	log_softmax_i = scores - np.max(scores) - np.log(np.sum(np.exp(scores - np.max(scores))))
	return log_softmax_i	

print(np.round(log_softmax([1, 1, 1]), 4))

[-1.0986 -1.0986 -1.0986]


***Sigmoid Activation Function Understanding***

In [ ]:
import math

def sigmoid(z: float) -> float:
	#Your code here
	pass

***Compute Multi-class Cross-Entropy Loss***

In [ ]:
import numpy as np

def compute_cross_entropy_loss(predicted_probs: np.ndarray, true_labels: np.ndarray, epsilon = 1e-15) -> float:
	pass

***Adam Optimizer***

In [ ]:
import numpy as np

def adam_optimizer(parameter, grad, m, v, t, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
	"""
	Update parameters using the Adam optimizer.
	Adjusts the learning rate based on the moving averages of the gradient and squared gradient.
	:param parameter: Current parameter value
	:param grad: Current gradient
	:param m: First moment estimate
	:param v: Second moment estimate
	:param t: Current timestep
	:param learning_rate: Learning rate (default=0.001)
	:param beta1: First moment decay rate (default=0.9)
	:param beta2: Second moment decay rate (default=0.999)
	:param epsilon: Small constant for numerical stability (default=1e-8)
	:return: tuple: (updated_parameter, updated_m, updated_v)
	"""
	# Your code here
	return np.round(parameter,5), np.round(m,5), np.round(v,5)

***Train Bigram Language Model as a Neural Network***

In [ ]:
import numpy as np

def train_bigram_nn(xs, ys, vocab_size, lr=10.0, num_iters=100, alpha=0.0):
	"""
	Train a one-layer neural-net bigram language model with full-batch gradient descent.

	Args:
		xs: list[int] of length N - input character indices in [0, vocab_size)
		ys: list[int] of length N - target next-character indices in [0, vocab_size)
		vocab_size: int V - number of distinct characters
		lr: float - learning rate
		num_iters: int - number of gradient descent steps
		alpha: float - L2 regularization strength on mean(W**2)

	Returns:
		list[list[float]] - trained weight matrix W of shape (V, V)
	"""
	pass

***Implement Weight Decay as L2 Regularization***

In [ ]:
def apply_weight_decay(parameters: list[list[float]], gradients: list[list[float]], 
					   lr: float, weight_decay: float, apply_to_all: list[bool]) -> list[list[float]]:
	"""
	Apply weight decay (L2 regularization) to parameters.
	
	Args:
		parameters: List of parameter arrays
		gradients: List of gradient arrays
		lr: Learning rate
		weight_decay: Weight decay factor
		apply_to_all: Boolean list indicating which parameter groups get weight decay
	
	Returns:
		Updated parameters
	"""
	# Your code here
	pass

***Build a Tokenizer for Language Modeling***